# A/B Significance Test - Ad vs PSA

Testing whether the ad campaign actually increases conversion compared to the psa control, using a two-proportion z-test, Wilson confidence intervals, and an effect size and power check.

In [1]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, proportion_confint, proportion_effectsize
from statsmodels.stats.power import NormalIndPower
from scipy.stats import norm

### test and helper functions

`run_ab_test` runs the z-test and builds Wilson confidence intervals for each group plus the difference, `effect_label` converts Cohen's h into a small/medium/large label, `achieved_power` checks how much power the actual sample sizes gave the test, and `required_n_per_group` answers the reverse question, how many users per group a smaller effect would need.

In [2]:
def run_ab_test(df, group_col="test group", outcome_col="converted", group_a="ad", group_b="psa", alpha=0.05):
    # z-critical value matches alpha, so diff_ci uses the same confidence
    # level as everything else in this function instead of always 95%
    z_crit = norm.ppf(1 - alpha / 2)
    a = df[df[group_col] == group_a][outcome_col]
    b = df[df[group_col] == group_b][outcome_col]

    n_a, n_b = len(a), len(b)
    conv_a, conv_b = a.sum(), b.sum()
    rate_a, rate_b = conv_a / n_a, conv_b / n_b

    z_stat, p_value = proportions_ztest([conv_a, conv_b], [n_a, n_b], alternative="two-sided")

    ci_a = proportion_confint(conv_a, n_a, alpha=alpha, method="wilson")
    ci_b = proportion_confint(conv_b, n_b, alpha=alpha, method="wilson")

    diff = rate_a - rate_b
    se_diff = np.sqrt(rate_a * (1 - rate_a) / n_a + rate_b * (1 - rate_b) / n_b)
    diff_ci = (diff - z_crit * se_diff, diff + z_crit * se_diff)

    effect_size_h = proportion_effectsize(rate_a, rate_b)

    return {
        "group_a": group_a, "group_b": group_b,
        "n_a": n_a, "n_b": n_b,
        "rate_a": rate_a, "rate_b": rate_b,
        "rate_a_ci": ci_a, "rate_b_ci": ci_b,
        "diff": diff, "diff_ci": diff_ci,
        "z_stat": z_stat, "p_value": p_value,
        "significant": p_value < alpha,
        "effect_size_h": effect_size_h,
        "alpha": alpha,
    }


In [3]:
def effect_label(h):
    h = abs(h)
    if h < 0.2:
        return "small"
    if h < 0.5:
        return "medium"
    return "large"


In [4]:
def print_result(r):
    print(f"{r['group_a']} (n={r['n_a']}): {r['rate_a']:.4%}  CI [{r['rate_a_ci'][0]:.4%}, {r['rate_a_ci'][1]:.4%}]")
    print(f"{r['group_b']} (n={r['n_b']}): {r['rate_b']:.4%}  CI [{r['rate_b_ci'][0]:.4%}, {r['rate_b_ci'][1]:.4%}]")
    print(f"diff: {r['diff']:.4%}  CI [{r['diff_ci'][0]:.4%}, {r['diff_ci'][1]:.4%}]")
    print(f"z = {r['z_stat']:.3f}, p = {r['p_value']:.4g}")
    print("significant" if r["significant"] else "not significant", f"at alpha={r['alpha']}")
    h = r["effect_size_h"]
    print(f"effect size (h) = {h:.4f} -> {effect_label(h)}")

In [5]:
def achieved_power(effect_size_h, n_a, n_b, alpha=0.05):
    analysis = NormalIndPower()
    ratio = n_b / n_a
    return analysis.power(effect_size=abs(effect_size_h), nobs1=n_a, alpha=alpha, ratio=ratio)


In [6]:
def required_n_per_group(min_effect_size_h, power=0.8, alpha=0.05):
    analysis = NormalIndPower()
    n = analysis.solve_power(effect_size=abs(min_effect_size_h), alpha=alpha, power=power, ratio=1.0)
    return int(np.ceil(n))

### results

Running the test on the real data.

In [7]:
df = pd.read_csv("../data/raw/marketing_AB.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

result = run_ab_test(df)
print_result(result)

power = achieved_power(result["effect_size_h"], result["n_a"], result["n_b"])
print(f"power: {power:.2%}")

ad (n=564577): 2.5547%  CI [2.5138%, 2.5961%]
psa (n=23524): 1.7854%  CI [1.6239%, 1.9627%]
diff: 0.7692%  CI [0.5951%, 0.9434%]
z = 7.370, p = 1.705e-13
significant at alpha=0.05
effect size (h) = 0.0530 -> small
power: 100.00%


### sample size check

The test found a small effect, checking how many users per group would be needed to reliably detect an effect that small at 80% power. Assumes equal group sizes, the real test was heavily imbalanced (96/4).

In [8]:
n_needed = required_n_per_group(min_effect_size_h=0.02, power=0.8)
print(f"users per group needed for h=0.02 at 80% power: {n_needed:,}")

users per group needed for h=0.02 at 80% power: 39,245
